# OpenOppsDB advanced usage

This read-only example shows durable joins, version history, sync observations, and when to use Parquet exports for columns that are projected out of the public SQLite preview copy.


In [ ]:
from pathlib import Path
import sqlite3

import pandas as pd


db_candidates = sorted(Path("/kaggle/input").glob("**/openoppsdb.sqlite"))
if not db_candidates:
    raise FileNotFoundError("No openoppsdb.sqlite input found under /kaggle/input")
DB_PATH = db_candidates[0]
DATASET_DIR = DB_PATH.parent
DB_URI = f"file:{DB_PATH}?mode=ro&immutable=1"
print(f"Reading OpenOppsDB snapshot from {DB_PATH}")


In [ ]:
with sqlite3.connect(DB_URI, uri=True) as conn:
    current_roles = pd.read_sql_query(
        """
        select
            j.id as job_id,
            coalesce(v.company, b.name) as company,
            v.title,
            j.provider_id,
            j.status,
            v.remote,
            v.employment_type,
            j.first_seen_at,
            j.last_seen_at,
            v.posting_url
        from jobs j
        join job_versions v on v.id = j.current_version_id
        left join boards b on b.key = j.board_key
        where j.status = 'open'
        order by j.last_seen_at desc
        limit 25
        """,
        conn,
    )

current_roles


In [ ]:
with sqlite3.connect(DB_URI, uri=True) as conn:
    version_history = pd.read_sql_query(
        """
        select
            j.id as job_id,
            coalesce(max(v.company), max(b.name)) as company,
            max(v.title) as latest_title,
            count(v.id) as version_count,
            min(v.first_seen_at) as first_version_seen_at,
            max(v.last_seen_at) as last_version_seen_at
        from jobs j
        join job_versions v on v.job_id = j.id
        left join boards b on b.key = j.board_key
        group by j.id
        having count(v.id) > 1
        order by version_count desc, last_version_seen_at desc
        limit 20
        """,
        conn,
    )
    observation_mix = pd.read_sql_query(
        """
        select observation_kind, count(*) as observations
        from job_sync_observations
        group by observation_kind
        order by observations desc
        """,
        conn,
    )

display(version_history)
observation_mix


In [ ]:
parquet_path = DATASET_DIR / "exports" / "parquet" / "job_versions.parquet"
if parquet_path.exists():
    try:
        full_text_sample = (
            pd.read_parquet(
                parquet_path,
                columns=["id", "title", "company", "description", "posting_url"],
            )
            .dropna(subset=["description"])
            .head(10)
        )
    except Exception as exc:
        full_text_sample = pd.DataFrame(
            {"note": [f"Parquet sample unavailable in this runtime: {type(exc).__name__}"]}
        )
else:
    full_text_sample = pd.DataFrame({"note": ["Parquet export not found"]})

full_text_sample
